In [5]:
# ============================================================
# FINAL MODEL EVALUATION
# Best Siamese U-Net + Caption Model
# ============================================================

import os
import json
import ast
import random
import numpy as np
import pandas as pd
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from nltk.translate.bleu_score import (
    corpus_bleu,
    SmoothingFunction
)

from nltk.translate.meteor_score import meteor_score

from rouge_score import rouge_scorer


# ============================================================
# 1. CONFIG
# ============================================================

CSV_PATH = r"C:\College\Dissertation\Data\FINAL_LEVIR_DATASET\final_dataset.csv"

CHECKPOINT = "best_siamese_unet_caption_model.pth"

IMAGE_SIZE = 256
MAX_LEN = 40
BATCH_SIZE = 4

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ============================================================
# 2. LOAD DATA
# ============================================================

df = pd.read_csv(
    CSV_PATH
)

val_df = df[
    df["split"] == "val"
].copy()

print(
    "\nValidation samples:",
    len(val_df)
)

print(
    val_df["changeflag"].value_counts()
)


# ============================================================
# 3. PARSE CAPTIONS
# ============================================================

def parse_captions(x):

    if isinstance(x, list):
        return x

    if pd.isna(x):
        return []

    try:
        return json.loads(x)
    except:

        try:
            return ast.literal_eval(x)

        except:
            return [str(x)]


val_df["captions"] = (
    val_df["captions"]
    .apply(parse_captions)
)


# ============================================================
# 4. LOAD VOCABULARY FROM CHECKPOINT
# ============================================================

checkpoint = torch.load(
    CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

word2idx = checkpoint[
    "word2idx"
]

idx2word = checkpoint[
    "idx2word"
]

PAD_IDX = word2idx[
    "<pad>"
]

START_IDX = word2idx[
    "<start>"
]

END_IDX = word2idx[
    "<end>"
]

UNK_IDX = word2idx[
    "<unk>"
]

VOCAB_SIZE = len(
    word2idx
)

print(
    "Vocabulary:",
    VOCAB_SIZE
)


# ============================================================
# 5. DATASET
# ============================================================

image_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor()
])


class LEVIRDataset(Dataset):

    def __init__(
        self,
        dataframe
    ):

        self.df = dataframe.reset_index(
            drop=True
        )


    def encode_caption(
        self,
        captions
    ):

        caption = captions[0]

        tokens = (
            caption
            .lower()
            .strip()
            .split()
        )

        tokens = (
            [START_IDX]
            +
            [
                word2idx.get(
                    token,
                    UNK_IDX
                )
                for token in tokens
            ]
            +
            [END_IDX]
        )

        if len(tokens) > MAX_LEN:

            tokens = tokens[
                :MAX_LEN
            ]

            tokens[-1] = END_IDX

        else:

            tokens += (
                [PAD_IDX]
                *
                (
                    MAX_LEN
                    -
                    len(tokens)
                )
            )

        return torch.tensor(
            tokens,
            dtype=torch.long
        )


    def __len__(self):

        return len(self.df)


    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        image_a = Image.open(
            row["cc_A"]
        ).convert("RGB")

        image_b = Image.open(
            row["cc_B"]
        ).convert("RGB")

        image_a = image_transform(
            image_a
        )

        image_b = image_transform(
            image_b
        )

        mask = Image.open(
            row["mask"]
        ).convert("L")

        mask = mask.resize(
            (
                IMAGE_SIZE,
                IMAGE_SIZE
            ),
            Image.Resampling.NEAREST
        )

        mask = np.asarray(
            mask,
            dtype=np.float32
        )

        mask = (
            mask > 127
        ).astype(
            np.float32
        )

        mask = torch.from_numpy(
            mask
        ).unsqueeze(0)

        caption = self.encode_caption(
            row["captions"]
        )

        changeflag = torch.tensor(
            float(row["changeflag"]),
            dtype=torch.float32
        )

        return {
            "image_a": image_a,
            "image_b": image_b,
            "mask": mask,
            "caption": caption,
            "changeflag": changeflag,
            "filename": row["filename"],
            "captions": row["captions"]
        }


val_dataset = LEVIRDataset(
    val_df
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)


# ============================================================
# 6. MODEL
# ============================================================

class DoubleConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                out_channels,
                out_channels,
                3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            )
        )


    def forward(self, x):

        return self.block(x)


class SiameseUNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.enc1 = DoubleConv(
            3, 32
        )

        self.enc2 = DoubleConv(
            32, 64
        )

        self.enc3 = DoubleConv(
            64, 128
        )

        self.enc4 = DoubleConv(
            128, 256
        )

        self.pool = nn.MaxPool2d(
            2
        )

        self.up3 = nn.ConvTranspose2d(
            256,
            128,
            2,
            stride=2
        )

        self.dec3 = DoubleConv(
            256,
            128
        )

        self.up2 = nn.ConvTranspose2d(
            128,
            64,
            2,
            stride=2
        )

        self.dec2 = DoubleConv(
            128,
            64
        )

        self.up1 = nn.ConvTranspose2d(
            64,
            32,
            2,
            stride=2
        )

        self.dec1 = DoubleConv(
            64,
            32
        )

        self.mask_head = nn.Conv2d(
            32,
            1,
            1
        )

        self.classifier = nn.Sequential(

            nn.AdaptiveAvgPool2d(1),

            nn.Flatten(),

            nn.Linear(
                256,
                64
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Dropout(
                0.3
            ),

            nn.Linear(
                64,
                1
            )
        )


    def encode(self, x):

        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool(e1)
        )

        e3 = self.enc3(
            self.pool(e2)
        )

        e4 = self.enc4(
            self.pool(e3)
        )

        return (
            e1,
            e2,
            e3,
            e4
        )


    def forward(
        self,
        image_a,
        image_b
    ):

        a1, a2, a3, a4 = (
            self.encode(image_a)
        )

        b1, b2, b3, b4 = (
            self.encode(image_b)
        )

        d1 = torch.abs(
            a1 - b1
        )

        d2 = torch.abs(
            a2 - b2
        )

        d3 = torch.abs(
            a3 - b3
        )

        d4 = torch.abs(
            a4 - b4
        )

        x = self.up3(d4)

        x = torch.cat(
            [x, d3],
            dim=1
        )

        x = self.dec3(x)

        x = self.up2(x)

        x = torch.cat(
            [x, d2],
            dim=1
        )

        x = self.dec2(x)

        x = self.up1(x)

        x = torch.cat(
            [x, d1],
            dim=1
        )

        x = self.dec1(x)

        mask = self.mask_head(x)

        classification = self.classifier(
            d4
        )

        return (
            mask,
            classification,
            d4
        )


class CaptionDecoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        feature_dim=256,
        embed_dim=128,
        hidden_dim=256
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=PAD_IDX
        )

        self.init_h = nn.Linear(
            feature_dim,
            hidden_dim
        )

        self.init_c = nn.Linear(
            feature_dim,
            hidden_dim
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_dim,
            vocab_size
        )


    def forward(
        self,
        features,
        captions
    ):

        pooled = F.adaptive_avg_pool2d(
            features,
            1
        ).flatten(1)

        h = self.init_h(
            pooled
        ).unsqueeze(0)

        c = self.init_c(
            pooled
        ).unsqueeze(0)

        inputs = captions[:, :-1]

        embeddings = self.embedding(
            inputs
        )

        outputs, _ = self.lstm(
            embeddings,
            (h, c)
        )

        logits = self.fc(
            outputs
        )

        return logits


class ChangeCaptionModel(nn.Module):

    def __init__(
        self,
        vocab_size
    ):

        super().__init__()

        self.siamese = SiameseUNet()

        self.caption_decoder = CaptionDecoder(
            vocab_size
        )


    def forward(
        self,
        image_a,
        image_b,
        captions
    ):

        mask, classification, features = (
            self.siamese(
                image_a,
                image_b
            )
        )

        caption_logits = (
            self.caption_decoder(
                features,
                captions
            )
        )

        return (
            mask,
            classification,
            caption_logits
        )


# ============================================================
# 7. LOAD BEST MODEL
# ============================================================

model = ChangeCaptionModel(
    VOCAB_SIZE
).to(DEVICE)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

model.eval()

print(
    "\nLoaded best checkpoint"
)

print(
    "Checkpoint epoch:",
    checkpoint["epoch"]
)

print(
    "Checkpoint val loss:",
    checkpoint["val_loss"]
)


# ============================================================
# 8. CAPTION GENERATION
# ============================================================

@torch.no_grad()
def generate_caption(
    model,
    image_a,
    image_b
):

    model.eval()

    _, _, features = model.siamese(
        image_a,
        image_b
    )

    pooled = F.adaptive_avg_pool2d(
        features,
        1
    ).flatten(1)

    h = model.caption_decoder.init_h(
        pooled
    ).unsqueeze(0)

    c = model.caption_decoder.init_c(
        pooled
    ).unsqueeze(0)

    current = torch.tensor(
        [[START_IDX]],
        dtype=torch.long,
        device=DEVICE
    )

    words = []

    for _ in range(
        MAX_LEN - 1
    ):

        embedding = (
            model.caption_decoder.embedding(
                current[:, -1:]
            )
        )

        output, (h, c) = (
            model.caption_decoder.lstm(
                embedding,
                (h, c)
            )
        )

        logits = model.caption_decoder.fc(
            output[:, -1, :]
        )

        next_token = (
            logits.argmax(
                dim=-1
            )
        )

        token_id = (
            next_token.item()
        )

        if token_id == END_IDX:
            break

        if token_id not in [
            PAD_IDX,
            START_IDX
        ]:

            words.append(
                idx2word.get(
                    token_id,
                    "<unk>"
                )
            )

        current = torch.cat(
            [
                current,
                next_token.unsqueeze(0)
            ],
            dim=1
        )

    return " ".join(words)


# ============================================================
# 9. EVALUATION
# ============================================================

all_true_flags = []
all_pred_flags = []

all_true_masks = []
all_pred_masks = []

references = []
hypotheses = []

prediction_rows = []

print(
    "\n" + "=" * 70
)

print(
    "EVALUATING BEST CHECKPOINT"
)

print(
    "=" * 70
)


with torch.no_grad():

    for batch in tqdm(
        val_loader,
        desc="Evaluating"
    ):

        image_a = batch[
            "image_a"
        ].to(
            DEVICE,
            non_blocking=True
        )

        image_b = batch[
            "image_b"
        ].to(
            DEVICE,
            non_blocking=True
        )

        masks = batch[
            "mask"
        ].to(
            DEVICE,
            non_blocking=True
        )

        flags = batch[
            "changeflag"
        ].to(
            DEVICE,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Model
        # ----------------------------------------------------

        pred_mask_logits, pred_flags, _ = (
            model.siamese(
                image_a,
                image_b
            )
        )

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        pred_probs = torch.sigmoid(
            pred_flags.squeeze(1)
        )

        pred_flags_binary = (
            pred_probs >= 0.5
        ).long()

        all_true_flags.extend(
            flags.cpu().numpy().astype(int)
        )

        all_pred_flags.extend(
            pred_flags_binary.cpu().numpy()
        )

        # ----------------------------------------------------
        # Segmentation
        # ----------------------------------------------------

        pred_masks = (
            torch.sigmoid(
                pred_mask_logits
            ) >= 0.5
        ).float()

        all_true_masks.append(
            masks.cpu()
        )

        all_pred_masks.append(
            pred_masks.cpu()
        )


# ============================================================
# 10. CLASSIFICATION METRICS
# ============================================================

accuracy = accuracy_score(
    all_true_flags,
    all_pred_flags
)

precision = precision_score(
    all_true_flags,
    all_pred_flags,
    zero_division=0
)

recall = recall_score(
    all_true_flags,
    all_pred_flags,
    zero_division=0
)

f1 = f1_score(
    all_true_flags,
    all_pred_flags,
    zero_division=0
)

cm = confusion_matrix(
    all_true_flags,
    all_pred_flags
)


print(
    "\n" + "=" * 70
)

print(
    "CHANGE CLASSIFICATION"
)

print(
    "=" * 70
)

print(
    f"Accuracy  : {accuracy:.4f}"
)

print(
    f"Precision : {precision:.4f}"
)

print(
    f"Recall    : {recall:.4f}"
)

print(
    f"F1        : {f1:.4f}"
)

print(
    "\nConfusion Matrix:"
)

print(cm)


# ============================================================
# 11. SEGMENTATION METRICS
# ============================================================

true_masks = torch.cat(
    all_true_masks,
    dim=0
)

pred_masks = torch.cat(
    all_pred_masks,
    dim=0
)

intersection = (
    true_masks * pred_masks
).sum(
    dim=(1, 2, 3)
)

union = (
    true_masks
    +
    pred_masks
    -
    true_masks * pred_masks
).sum(
    dim=(1, 2, 3)
)

true_area = true_masks.sum(
    dim=(1, 2, 3)
)

pred_area = pred_masks.sum(
    dim=(1, 2, 3)
)

dice = (
    2 * intersection + 1e-7
) / (
    true_area
    +
    pred_area
    +
    1e-7
)

iou = (
    intersection + 1e-7
) / (
    union + 1e-7
)


# Pixel accuracy

pixel_accuracy = (
    true_masks == pred_masks
).float().mean().item()


print(
    "\n" + "=" * 70
)

print(
    "SEGMENTATION"
)

print(
    "=" * 70
)

print(
    f"IoU             : {iou.mean().item():.4f}"
)

print(
    f"Dice            : {dice.mean().item():.4f}"
)

print(
    f"Pixel Accuracy  : {pixel_accuracy:.4f}"
)


# ============================================================
# 12. CAPTION GENERATION
# ============================================================

print(
    "\nGenerating captions..."
)

references = []
hypotheses = []

prediction_rows = []

for idx in tqdm(
    range(len(val_dataset)),
    desc="Captions"
):

    sample = val_dataset[idx]

    image_a = sample[
        "image_a"
    ].unsqueeze(0).to(
        DEVICE
    )

    image_b = sample[
        "image_b"
    ].unsqueeze(0).to(
        DEVICE
    )

    prediction = generate_caption(
        model,
        image_a,
        image_b
    )

    refs = [
        caption.lower()
        .strip()
        .split()
        for caption
        in sample["captions"]
    ]

    hyp = (
        prediction
        .lower()
        .strip()
        .split()
    )

    references.append(
        refs
    )

    hypotheses.append(
        hyp
    )

    prediction_rows.append({

        "filename":
            sample["filename"],

        "changeflag":
            int(
                sample[
                    "changeflag"
                ].item()
            ),

        "prediction":
            prediction,

        "ground_truth":
            " | ".join(
                sample["captions"]
            )
    })


# ============================================================
# 13. BLEU
# ============================================================

smooth = SmoothingFunction().method1

bleu1 = corpus_bleu(
    references,
    hypotheses,
    weights=(1, 0, 0, 0),
    smoothing_function=smooth
)

bleu2 = corpus_bleu(
    references,
    hypotheses,
    weights=(0.5, 0.5, 0, 0),
    smoothing_function=smooth
)

bleu3 = corpus_bleu(
    references,
    hypotheses,
    weights=(1/3, 1/3, 1/3, 0),
    smoothing_function=smooth
)

bleu4 = corpus_bleu(
    references,
    hypotheses,
    weights=(0.25, 0.25, 0.25, 0.25),
    smoothing_function=smooth
)


# ============================================================
# 14. METEOR
# ============================================================

meteor_scores = []

for refs, hyp in zip(
    references,
    hypotheses
):

    if len(hyp) == 0:

        meteor_scores.append(
            0.0
        )

    else:

        meteor_scores.append(
            meteor_score(
                refs,
                hyp
            )
        )


meteor = np.mean(
    meteor_scores
)


# ============================================================
# 15. ROUGE-L
# ============================================================

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

rouge_scores = []

for refs, hyp in zip(
    references,
    hypotheses
):

    hyp_text = " ".join(
        hyp
    )

    scores = []

    for ref in refs:

        ref_text = " ".join(
            ref
        )

        scores.append(
            rouge.score(
                ref_text,
                hyp_text
            )["rougeL"].fmeasure
        )

    rouge_scores.append(
        max(scores)
    )


rouge_l = np.mean(
    rouge_scores
)


# ============================================================
# 16. CAPTION RESULTS
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "CAPTION GENERATION"
)

print(
    "=" * 70
)

print(
    f"BLEU-1  : {bleu1:.4f}"
)

print(
    f"BLEU-2  : {bleu2:.4f}"
)

print(
    f"BLEU-3  : {bleu3:.4f}"
)

print(
    f"BLEU-4  : {bleu4:.4f}"
)

print(
    f"METEOR  : {meteor:.4f}"
)

print(
    f"ROUGE-L : {rouge_l:.4f}"
)


# ============================================================
# 17. SAVE PREDICTIONS
# ============================================================

predictions_df = pd.DataFrame(
    prediction_rows
)

predictions_df.to_csv(
    "final_model_predictions.csv",
    index=False
)


# ============================================================
# 18. SAVE METRICS
# ============================================================

metrics = {

    "classification_accuracy":
        accuracy,

    "classification_precision":
        precision,

    "classification_recall":
        recall,

    "classification_f1":
        f1,

    "segmentation_iou":
        iou.mean().item(),

    "segmentation_dice":
        dice.mean().item(),

    "segmentation_pixel_accuracy":
        pixel_accuracy,

    "bleu1":
        bleu1,

    "bleu2":
        bleu2,

    "bleu3":
        bleu3,

    "bleu4":
        bleu4,

    "meteor":
        meteor,

    "rougeL":
        rouge_l
}

metrics_df = pd.DataFrame(
    [metrics]
)

metrics_df.to_csv(
    "final_model_metrics.csv",
    index=False
)


# ============================================================
# 19. FINAL SUMMARY
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "FINAL MODEL RESULTS"
)

print(
    "=" * 70
)

print(
    f"Classification F1 : {f1:.4f}"
)

print(
    f"Segmentation IoU  : {iou.mean().item():.4f}"
)

print(
    f"Segmentation Dice : {dice.mean().item():.4f}"
)

print(
    f"BLEU-1            : {bleu1:.4f}"
)

print(
    f"BLEU-2            : {bleu2:.4f}"
)

print(
    f"BLEU-3            : {bleu3:.4f}"
)

print(
    f"BLEU-4            : {bleu4:.4f}"
)

print(
    f"METEOR            : {meteor:.4f}"
)

print(
    f"ROUGE-L           : {rouge_l:.4f}"
)

print("\nSaved:")
print("  final_model_predictions.csv")
print("  final_model_metrics.csv")

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU

Validation samples: 919
changeflag
1    467
0    452
Name: count, dtype: int64
Vocabulary: 899

Loaded best checkpoint
Checkpoint epoch: 67
Checkpoint val loss: 1.7156884965689285

EVALUATING BEST CHECKPOINT


Evaluating:   0%|          | 0/230 [00:00<?, ?it/s]


CHANGE CLASSIFICATION
Accuracy  : 0.8411
Precision : 0.8309
Recall    : 0.8630
F1        : 0.8466

Confusion Matrix:
[[370  82]
 [ 64 403]]

SEGMENTATION
IoU             : 0.1186
Dice            : 0.1186
Pixel Accuracy  : 0.9560

Generating captions...


Captions:   0%|          | 0/919 [00:00<?, ?it/s]


CAPTION GENERATION
BLEU-1  : 0.7535
BLEU-2  : 0.6350
BLEU-3  : 0.5466
BLEU-4  : 0.4754
METEOR  : 0.6793
ROUGE-L : 0.6656

FINAL MODEL RESULTS
Classification F1 : 0.8466
Segmentation IoU  : 0.1186
Segmentation Dice : 0.1186
BLEU-1            : 0.7535
BLEU-2            : 0.6350
BLEU-3            : 0.5466
BLEU-4            : 0.4754
METEOR            : 0.6793
ROUGE-L           : 0.6656

Saved:
  final_model_predictions.csv
  final_model_metrics.csv
